In [7]:
import pandas as pd
import numpy as np
import re
import math
import pickle

from pathlib import Path
from urllib.parse import urlparse

BASE_DIR = Path.cwd()

# If notebook is inside /notebooks, move one folder back to project root
if not (BASE_DIR / "data").exists():
    BASE_DIR = Path("..").resolve()

DATA_PATH = BASE_DIR / "data" / "processed" / "final_phishing_dataset.csv"
OUTPUT_PATH = BASE_DIR / "models" / "ml_features_dataset.csv"

print("Project folder:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())
print("Output path:", OUTPUT_PATH)

Project folder: D:\phishing-detection
Dataset path: D:\phishing-detection\data\processed\final_phishing_dataset.csv
Dataset exists: True
Output path: D:\phishing-detection\models\ml_features_dataset.csv


In [8]:
df = pd.read_csv(DATA_PATH)

df = df[["url", "label"]].dropna()
df["url"] = df["url"].astype(str).str.strip().str.lower()
df = df[df["url"] != ""]

print(df.shape)
print(df["label"].value_counts())
df.head()

(99995, 2)
label
0    50000
1    49995
Name: count, dtype: int64


,url,label
0,https://docs.google.com/presentation/d/1isueil...,1
1,https://bit.ly/3c0lhvd,1
2,https://www.hobbies.net/dynaflite/,0
3,https://calm-place-199089.framer.app/,1
4,https://stream-billing.com/login,1


In [9]:
def calculate_entropy(url):
    if len(url) == 0:
        return 0
    
    probabilities = [url.count(c) / len(url) for c in set(url)]
    return -sum(p * math.log2(p) for p in probabilities)


def has_ip_address(hostname):
    pattern = r"^\d{1,3}(\.\d{1,3}){3}$"
    return 1 if re.match(pattern, hostname) else 0


def extract_features(url):
    url = str(url).strip().lower()

    if not url.startswith(("http://", "https://")):
        url = "https://" + url

    parsed = urlparse(url)

    hostname = parsed.netloc
    path = parsed.path
    query = parsed.query

    domain_parts = hostname.split(".")
    tld = domain_parts[-1] if len(domain_parts) > 1 else ""

    suspicious_tlds = ["tk", "ml", "ga", "cf", "gq", "xyz", "top", "club"]

    features = {
        "url_length": len(url),
        "host_length": len(hostname),
        "path_length": len(path),
        "query_length": len(query),

        "count_dots": url.count("."),
        "count_hyphen": url.count("-"),
        "count_at": url.count("@"),
        "count_qmark": url.count("?"),
        "count_equal": url.count("="),
        "count_slash": url.count("/"),
        "count_digits": sum(char.isdigit() for char in url),

        "has_https": 1 if parsed.scheme == "https" else 0,
        "has_ip": has_ip_address(hostname),
        "subdomain_count": max(len(domain_parts) - 2, 0),
        "tld_suspicious": 1 if tld in suspicious_tlds else 0,
        "entropy": calculate_entropy(url),

        "has_login": 1 if "login" in url else 0,
        "has_verify": 1 if "verify" in url else 0,
        "has_secure": 1 if "secure" in url else 0,
        "has_update": 1 if "update" in url else 0,
        "has_bank": 1 if "bank" in url else 0,
    }

    return features

In [10]:
features_df = df["url"].apply(extract_features).apply(pd.Series)

features_df["label"] = df["label"].astype(int)

print(features_df.shape)
print(features_df["label"].value_counts())
features_df.head()

(99995, 22)
label
0    50000
1    49995
Name: count, dtype: int64


,url_length,host_length,path_length,query_length,count_dots,count_hyphen,count_at,count_qmark,count_equal,count_slash,...,has_ip,subdomain_count,tld_suspicious,entropy,has_login,has_verify,has_secure,has_update,has_bank,label
0,137.0,15.0,64.0,49.0,3.0,0.0,0.0,1.0,4.0,6.0,...,0.0,1.0,0.0,4.821553,0.0,0.0,0.0,0.0,0.0,1
1,22.0,6.0,8.0,0.0,1.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,3.845351,0.0,0.0,0.0,0.0,0.0,1
2,34.0,15.0,11.0,0.0,2.0,0.0,0.0,0.0,0.0,4.0,...,0.0,1.0,0.0,3.984234,0.0,0.0,0.0,0.0,0.0,0
3,37.0,28.0,1.0,0.0,2.0,2.0,0.0,0.0,0.0,3.0,...,0.0,1.0,0.0,4.087568,0.0,0.0,0.0,0.0,0.0,1
4,32.0,18.0,6.0,0.0,1.0,1.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,4.093139,1.0,0.0,0.0,0.0,0.0,1


In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

features_df.to_csv(OUTPUT_PATH, index=False)

print("ML feature dataset saved successfully:")
print(OUTPUT_PATH)

ML feature dataset saved successfully:
D:\phishing-detection\models\ml_features_dataset.csv


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

import pickle

In [13]:
FEATURE_DATA_PATH = BASE_DIR / "models" / "ml_features_dataset.csv"

ml_df = pd.read_csv(FEATURE_DATA_PATH)

X = ml_df.drop(columns=["label"])
y = ml_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

Train shape: (79996, 21)
Test shape: (19999, 21)
label
0    40000
1    39996
Name: count, dtype: int64
label
0    10000
1     9999
Name: count, dtype: int64


In [14]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with open(BASE_DIR / "models" / "ml_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Scaler saved.")

Scaler saved.


In [15]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)

print(classification_report(y_test, lr_pred))
print(confusion_matrix(y_test, lr_pred))

              precision    recall  f1-score   support

           0       0.76      0.79      0.77     10000
           1       0.78      0.75      0.76      9999

    accuracy                           0.77     19999
   macro avg       0.77      0.77      0.77     19999
weighted avg       0.77      0.77      0.77     19999

[[7876 2124]
 [2545 7454]]


In [16]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96     10000
           1       0.97      0.96      0.96      9999

    accuracy                           0.96     19999
   macro avg       0.96      0.96      0.96     19999
weighted avg       0.96      0.96      0.96     19999

[[9712  288]
 [ 431 9568]]


In [17]:
svm_model = LinearSVC(random_state=42)

svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)

print(classification_report(y_test, svm_pred))
print(confusion_matrix(y_test, svm_pred))

              precision    recall  f1-score   support

           0       0.74      0.83      0.78     10000
           1       0.81      0.70      0.75      9999

    accuracy                           0.77     19999
   macro avg       0.77      0.77      0.77     19999
weighted avg       0.77      0.77      0.77     19999

[[8328 1672]
 [2962 7037]]


In [18]:
results = []

def add_results(model_name, y_true, y_pred):
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred)
    })

add_results("Logistic Regression", y_test, lr_pred)
add_results("Random Forest", y_test, rf_pred)
add_results("Linear SVM", y_test, svm_pred)

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.766538,0.778242,0.745475,0.761506
1,Random Forest,0.964048,0.970779,0.956896,0.963787
2,Linear SVM,0.768288,0.808015,0.703770,0.752298


In [19]:
with open(BASE_DIR / "models" / "logistic_regression.pkl", "wb") as f:
    pickle.dump(lr_model, f)

with open(BASE_DIR / "models" / "random_forest.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open(BASE_DIR / "models" / "linear_svm.pkl", "wb") as f:
    pickle.dump(svm_model, f)

results_df.to_csv(BASE_DIR / "models" / "ml_model_results.csv", index=False)

print("All ML models and results saved in models folder.")

All ML models and results saved in models folder.
